# 2 · MiniPlace: an orchestrator steers the team

**Command A+ coordinator · North Mini Code workers · one shared 384×288 canvas**

The coordinator invents a division of work to reconstruct the Canadian flag and Cohere logo. Delegation returns immediately, letting the coordinator observe, communicate, redirect, and cancel jobs while workers continue.

Assignments are instructions, not enforced territory. The models choose their actions; Python verifies every actual pixel.

## 1. Setup

From the project folder, run `uv sync --locked` and `uv run jupyter lab`. Use the Python 3 kernel and run the cells in order. Restart the kernel after updating the helper files.

For Colab, upload the three `miniplace_*.py` helpers and the `assets/` folder alongside this notebook. The install cell below installs the required libraries.

Provide a [Cohere API key](https://dashboard.cohere.com/api-keys) with access to `command-a-plus-05-2026` and `north-mini-code-1-0`, using `COHERE_API_KEY`, `CO_API_KEY`, or the hidden prompt. Both roles receive text-only inputs.

In [ ]:
import shutil
import subprocess
import sys
from importlib.metadata import PackageNotFoundError, version
try:
    sdk_ready = version('cohere') == '7.1.1' and all(version(package) for package in ('anywidget', 'jsonschema'))
except PackageNotFoundError:
    sdk_ready = False
if not sdk_ready:
    uv_binary = shutil.which('uv')
    if uv_binary:
        command = [uv_binary]
    else:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'uv'], check=True)
        command = [sys.executable, '-m', 'uv']
    subprocess.run(command + ['pip', 'install', '--python', sys.executable, 'cohere==7.1.1', 'anywidget>=0.9,<1', 'jsonschema>=4.23,<5'], check=True)

In [ ]:
import os
from getpass import getpass
import cohere
try:
    from google.colab import output
    output.enable_custom_widget_manager()
except ImportError:
    pass
from miniplace_runtime import Config, Dashboard, Studio, load_cohere_mural

api_key = os.getenv('COHERE_API_KEY') or os.getenv('CO_API_KEY') or getpass('Cohere API key: ')
if not api_key.strip():
    raise ValueError('Enter a Cohere API key.')
client = cohere.AsyncClientV2(api_key=api_key.strip(), timeout=60, max_retries=0)
del api_key

## 2. Shared goal, budgets, and guidance

The local reference has **110,592 pixels and 14 colors**, with **60,432 incorrect pixels** on the blank starting board.

With `full_team_start=True`, the coordinator's first action supplies one model-authored task per worker. The complete roster is validated before all jobs are scheduled together. Later decisions can refill idle workers and redirect active ones.

Each agent retains its full conversation in memory. Active requests keep recent complete exchanges and structured memory after the context threshold; `recall_context` retrieves earlier coordination.

In [ ]:
config = Config(
    async_paint=True, raw_paint_tools=False, max_pixels_per_action=4096, drain_seconds=None,
    model='north-mini-code-1-0', coordinator_model='command-a-plus-05-2026',
    thinking_budget=0, coordinator_thinking_budget=1024,
    context_soft_limit=100_000, context_recent_turns=4,
    painters=16, worker_concurrency=16, full_team_start=True,
    max_agent_calls=500, max_coordinator_calls=500, max_seconds=360*4,
    pixel_delay=0.001, ui_fps=16,
)
target, palette = load_cohere_mural()
COORDINATOR_GUIDANCE = (
    'Start with launch_team: provide one concise task for every worker in a single full-roster launch. '
    'Use the text reference overview and error counts to plan; each worker can inspect its own area in detail. '
    'Invent comparable-sized independent end-to-end reconstruction jobs, with clear bounds and a match-the-reference completion criterion. '
    'Workers should inspect as needed AND paint/verify their scope. Avoid inspection-only sweeps, placeholder idle tasks, '
    'and one-pixel micromanagement while substantial unfinished regions remain. Batch refills for available workers. '
    'Check workforce.available_workers and workforce.never_started_workers on every decision. '
    'Promptly redelegate unfinished work to idle workers while others continue. '
    'Keep productive jobs stable, repair bottlenecks, and taper the crew only when remaining work becomes scarce. '
    'Completion requires EVERY pixel correct and accepted finish_mural verification. Never describe partial progress '
    'as complete, essentially complete, or acceptable accuracy. If limits stop work, report INCOMPLETE with the exact remaining errors.'
)
PAINTER_GUIDANCE = (
    'Match the actual reference throughout your current task. Announce a useful bounded scope and paint_work its returned work_id promptly. '
    'Each work ID paints its entire scope asynchronously; do not requeue it or split an accepted item into tiny strips. '
    'Use current progress and work_advice, and continue through as many bounded work items as the task needs. '
    'Avoid already-correct areas. Verify the whole delegated task before reporting done; '
    'tell the coordinator exactly what remains if you are blocked or near your decision limit.'
)
preview_studio = Studio('hierarchy', config=config, target=target, palette=palette)
preview = Dashboard(preview_studio)  # Reference preview; no model calls.

## 3. Tools for the coordinator

| Tool | Effect |
|---|---|
| `launch_team` | Submit one task per worker and start the full roster together |
| `delegate` | Start or redirect named workers; return job handles immediately |
| `inspect_canvas` / `watch_canvas` | Observe progress while workers continue |
| `send_message` / `post_update` | Communicate instructions, milestones, and handoffs |
| `steer` | Redirect a worker during inference or painting |
| `cancel_agent` | Stop one job while the others continue |
| `finish_mural` | Submit the entire board for exact verification |

Workers choose bounded scopes with `announce_work`, then paint them asynchronously using `paint_work(work_id)`. A delegated task can span several work items. Each worker can communicate or plan while its brush queue runs.

For a raw-painting experiment, `raw_paint_tools=True` exposes standalone inspection and custom pixel tools; `reference_brush=False` requires pixel transcription.

In [ ]:
[tool['function']['name'] for tool in preview_studio.tools_for('Coordinator')]

## 4. Run the demo

Watch for full-team startup, useful task boundaries, and rolling reassignment. **Active worker jobs**, **API requests**, and **painting brushes** are distinct measures.

Use **Full screen**, **Work board**, and **Expand graph** to follow the team. Select an agent to see its task, request phase, last-output age, and deadline. Scrolling up pauses log following; **Resume live** returns to new events.

Urgent steering interrupts a brush at its next pixel yield. A response to an obsolete instruction may arrive, but its actions are skipped. With `interrupt=False`, the current decision can finish first.

Each API request has a 60-second total deadline by default (`request_timeout`), with bounded retries. Only complete, validated tool calls execute.

In [ ]:
hierarchy_studio = Studio('hierarchy', client=client, config=config, target=target, palette=palette)
hierarchy_studio.extra_prompts['Coordinator'] = COORDINATOR_GUIDANCE
hierarchy_studio.extra_prompts['hierarchy'] = PAINTER_GUIDANCE
hierarchy_result = await hierarchy_studio.run()

In [ ]:
hierarchy_studio.summary()
check = hierarchy_studio.check()
{key: check[key] for key in ('valid', 'matched', 'total', 'wrong')}

## 5. Explore the result

- Inspect `hierarchy_studio.controls`, `hierarchy_studio.messages`, and `hierarchy_studio.plan_history` to see how assignments evolved.
- Replay recent frames with `await hierarchy_studio.view.replay()`; this makes no model calls.
- Edit the guidance and start a fresh run to compare supervision strategies.

The run stops on exact completion or its configured limits. The pixel check reports an incomplete result honestly.